# Notebook to do assignment of classes

In [37]:
ROOT= '../'

In [38]:
!pip install wildlife-datasets git+https://github.com/WildlifeDatasets/wildlife-tools --quiet --upgrade-strategy only-if-needed

In [39]:
import shutil, os
import warnings
warnings.filterwarnings('ignore')
import os

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances

Config Parameters

In [82]:
import torch

CFG = dict(
    model_name        = 'vit_base_patch14_reg4_dinov2',
    img_size          = 224,
    embed_dim         = 512,      # ViT-Base output dim; set smaller (e.g. 512) to project down
    arcface_s         = 16.0,     # scale
    arcface_m         = 0.35,     # margin
    batch_size        = 32,
    num_epochs        = 30,
    lr_head           = 1e-3,     # head + projector LR during warmup
    lr_backbone       = 2e-5,     # last 4 ViT blocks LR during fine-tune
    warmup_epochs     = 3,        # freeze backbone for this many epochs first
    mls_threshold     = 0.564,     # auto-calibrated from train MLS distribution
    dbscan_eps        = 0.45,      # cosine distance cutoff for DBSCAN
    mls_pct           = 5,        # percentile of train MLS used as threshold
    dbscan_min_samples= 2,
    device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'),
    seed              = 42,
    unfrozen_blocks   = 4,
    weight_decay  = 5e-4,
)
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print('Device:', CFG['device'])

Device: mps


## Data Loading 

In [42]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit

meta = pd.read_csv(os.path.join(ROOT, 'metadata.csv'))

# All training images with known identity labels
all_train_df = meta[(meta['split'] == 'train') & (meta['identity'].notna())].copy()
query_df     = meta[meta['split'] == 'test'].copy()

# Split by identity group: ~15% of identities go to val (their images are "novel" to the model)
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=CFG['seed'])
train_idx, val_idx = next(gss.split(all_train_df, groups=all_train_df['identity']))
train_meta = all_train_df.iloc[train_idx].reset_index(drop=True)
val_meta   = all_train_df.iloc[val_idx].reset_index(drop=True)

# Fit LabelEncoder on train identities only
le = LabelEncoder()
le.fit(train_meta['identity'].values)

NUM_CLASSES = len(le.classes_)
print(f'Known identities (train classes): {NUM_CLASSES}')
print(f'Train images : {len(train_meta)}')
print(f'Val images   : {len(val_meta)}  ({val_meta["identity"].nunique()} novel identities)')
print(f'Query images : {len(query_df)}')

Known identities (train classes): 936
Train images : 11578
Val images   : 1496  (166 novel identities)
Query images : 2409


In [43]:
class TrainDataset(Dataset):
    """Returns (image_tensor, integer_label) for ArcFace training."""
    def __init__(self, metadata: pd.DataFrame, le: LabelEncoder,
                 transform, root: str):
        self.paths     = metadata['path'].values
        self.labels    = le.transform(metadata['identity'].values)
        self.transform = transform
        self.root      = root
        self.le        = le

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.paths[idx])).convert('RGB')
        return self.transform(img), self.labels[idx]

    @property
    def num_classes(self):
        return len(self.le.classes_)


class ValDataset(Dataset):
    """
    Returns (image_tensor, identity_string) for open-set evaluation.
    Contains a mix of known individuals (identity in train) and novel
    individuals (identity never seen in training).
    """
    def __init__(self, metadata: pd.DataFrame, transform, root: str):
        self.paths      = metadata['path'].values
        self.identities = metadata['identity'].fillna('unknown').values
        self.transform  = transform
        self.root       = root

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.paths[idx])).convert('RGB')
        return self.transform(img), self.identities[idx]


size = CFG['img_size']
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.Resize((size, size)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])
val_tf = T.Compose([
    T.Resize((size, size)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

num_workers = 0  # 0 for Colab; set to 4 for local

train_ds  = TrainDataset(train_meta, le, train_tf, ROOT)
val_ds    = ValDataset(val_meta,         val_tf,   ROOT)
query_ds  = ValDataset(query_df,         val_tf,   ROOT)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=num_workers, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=num_workers)
query_loader = DataLoader(query_ds, batch_size=64, shuffle=False, num_workers=num_workers)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Query batches: {len(query_loader)}')

Train batches: 361  |  Val batches: 24  |  Query batches: 38


## Model

In [44]:
class ArcFaceLoss(nn.Module):
    """Additive Angular Margin Loss.
    Adds a fixed margin to the target class angle before softmax,
    pushing intra-class embeddings together and inter-class apart.
    """
    def __init__(self, in_features, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))  # (B, C)
        sine   = (1.0 - cosine.pow(2)).clamp(0, 1).sqrt()
        phi    = cosine * self.cos_m - sine * self.sin_m  # cos(theta + margin)
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.view(-1, 1), 1.0)
        logits  = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return F.cross_entropy(logits, labels, label_smoothing=0.1), logits

In [45]:
class DINOv2ReID(nn.Module):
    def __init__(self, model_name, embed_dim, num_classes, s, m):
        super().__init__()
        # num_classes=0, backbone returns raw [CLS] token embedding
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        bb_dim = self.backbone.num_features  # 768 for ViT-Base
        # Optional linear projection; keeps dim if embed_dim == bb_dim
        self.projector = (
            nn.Sequential(nn.Linear(bb_dim, embed_dim), nn.BatchNorm1d(embed_dim))
            if embed_dim != bb_dim else nn.Identity()
        )
        self.arcface = ArcFaceLoss(embed_dim, num_classes, s=s, m=m)

    def embed(self, x):
        """Return L2-normalised embedding (used at inference)."""
        return F.normalize(self.projector(self.backbone(x)), dim=1)

    def forward(self, x, labels=None):
        emb = self.embed(x)
        if labels is not None:
            loss, logits = self.arcface(emb, labels)
            return loss, logits, emb
        return emb

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_blocks(self, n=4):
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in list(self.backbone.blocks)[-n:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True


model = DINOv2ReID(
    model_name=CFG['model_name'],
    embed_dim=CFG['embed_dim'],
    num_classes=NUM_CLASSES,
    s=CFG['arcface_s'],
    m=CFG['arcface_m'],                           
).to(CFG['device'])

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Total params: {total_params:.1f}M')

Total params: 86.7M


## load previous checkpoint

In [46]:
ckpt_path = os.path.join(ROOT, 'checkpoints/dinov2/dinov2_7.pt')

In [47]:
# Load the saved model checkpoint
checkpoint = torch.load(ckpt_path, map_location=CFG['device'], weights_only=False)

# Recreate the model with saved config
model_cfg = checkpoint['cfg']
model = DINOv2ReID(
    model_name=model_cfg['model_name'],
    embed_dim=model_cfg['embed_dim'],
    num_classes=len(checkpoint['le'].classes_),
    s=model_cfg['arcface_s'],
    m=model_cfg['arcface_m'],
).to(CFG['device'])

# Load the state dict
model.load_state_dict(checkpoint['model_state'])
model.eval()

# Recreate label encoder if needed for mapping
le = checkpoint['le']
idx_to_class = {i: cls for i, cls in enumerate(le.classes_)}

print(f'Loaded model from {ckpt_path}')

Loaded model from ../checkpoints/dinov2/dinov2_7.pt


## Detection

In [48]:
@torch.no_grad()
def extract_features(model, loader, device):
    """Returns (embeddings, raw_cosine_logits, labels) as numpy arrays."""
    model.eval()
    embs, logits_all, labels_all = [], [], []
    for imgs, labels in loader:
        imgs  = imgs.to(device)
        emb   = model.embed(imgs)                                           # (B, D)
        # Raw cosine logits — no margin, used for MLS scoring
        logit = F.linear(emb, F.normalize(model.arcface.weight))            # (B, C)
        embs.append(emb.cpu())
        logits_all.append(logit.cpu())
        # labels may be a tensor (TrainDataset) or tuple of strings (ValDataset)
        if isinstance(labels, torch.Tensor):
            labels_all.append(labels)
        else:
            labels_all.extend(labels)
    embs_np   = torch.cat(embs).numpy()
    logits_np = torch.cat(logits_all).numpy()
    labels_np = torch.cat(labels_all).numpy() if isinstance(labels_all[0], torch.Tensor) else np.array(labels_all)
    return embs_np, logits_np, labels_np


print('Extracting train embeddings...')
train_embs, train_logits, train_labels = extract_features(model, train_loader, CFG['device'])

print('Extracting query embeddings...')
query_embs, query_logits, query_labels = extract_features(model, query_loader, CFG['device'])

print(f'train_embs : {train_embs.shape}')
print(f'query_embs : {query_embs.shape}')

Extracting train embeddings...
Extracting query embeddings...
train_embs : (11552, 512)
query_embs : (2409, 512)


In [56]:
train_mls = train_logits.max(axis=1)  # (N_train,)
query_mls = query_logits.max(axis=1)  # (N_query,)

In [70]:
# ── Assign known predictions ──────────────────────────────────────────────────
is_known = query_mls >= CFG['mls_threshold']
print(f'Classified as known  : {is_known.sum()} / {len(is_known)}')
print(f'Classified as unknown: {(~is_known).sum()} / {len(is_known)}')

predicted_ids = np.full(len(query_df), 'unknown', dtype=object)

if is_known.any():
    pred_cls = query_logits[is_known].argmax(axis=1)
    predicted_ids[is_known] = [idx_to_class[c] for c in pred_cls]

Classified as known  : 694 / 2409
Classified as unknown: 1715 / 2409


In [79]:
print(CFG['mls_threshold'])
print(CFG['dbscan_eps'])

0.564
0.45


In [80]:
# ── DBSCAN in cosine-distance space ──────────────────────────────────────────
unknown_mask = ~is_known
unknown_embs = query_embs[unknown_mask]

if unknown_embs.shape[0] > 0:
    dist_matrix = pairwise_distances(unknown_embs, metric='cosine')
    db = DBSCAN(
        eps=CFG['dbscan_eps'],
        min_samples=CFG['dbscan_min_samples'],
        metric='precomputed',
    ).fit(dist_matrix)

    cluster_labels = db.labels_
    n_clusters = len(set(cluster_labels) - {-1})
    n_noise    = (cluster_labels == -1).sum()
    print(f'DBSCAN clusters: {n_clusters}  |  noise singletons: {n_noise}')

    unknown_indices = np.where(unknown_mask)[0]
    dataset_col     = query_df['dataset'].values  # e.g. 'LynxID2025', 'SeaTurtleID2022'

    dataset_counters = {}  # tracks next free index per dataset
    cluster_to_id    = {}  # maps DBSCAN cluster label → assigned identity string

    for idx, cl in zip(unknown_indices, cluster_labels):
        dataset = dataset_col[idx]
        if cl == -1:
            # Singleton: unique ID for this image within its dataset
            n = dataset_counters.get(dataset, 0)
            predicted_ids[idx] = f'{dataset}_unknown_{n}'
            dataset_counters[dataset] = n + 1
        else:
            # Cluster: all images in this group share the same ID
            if cl not in cluster_to_id:
                n = dataset_counters.get(dataset, 0)
                cluster_to_id[cl] = f'{dataset}_unknown_{n}'
                dataset_counters[dataset] = n + 1
            predicted_ids[idx] = cluster_to_id[cl]
else:
    print('No unknown samples detected — lower mls_threshold if unexpected.')

DBSCAN clusters: 107  |  noise singletons: 675


In [54]:
for eps in [0.15, 0.20, 0.25, 0.30, 0.35]:
    db = DBSCAN(eps=eps, min_samples=CFG['dbscan_min_samples'], metric='cosine')
    labels = db.fit_predict(unknown_embs)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"eps={eps}  clusters={n_clusters}  singletons={n_noise}")


eps=0.15  clusters=33  singletons=1517
eps=0.2  clusters=55  singletons=1343
eps=0.25  clusters=81  singletons=1239
eps=0.3  clusters=106  singletons=1116
eps=0.35  clusters=123  singletons=988


## formatting submission

In [81]:
import re

def clean_cluster_id(pid):
    """Normalize identity strings to dataset_number format.
    e.g. LynxID2025_lynx_25 -> LynxID2025_25
         SeaTurtleID2022_t323 -> SeaTurtleID2022_323
         SalamanderID2025_131 -> SalamanderID2025_131 (unchanged)
         unknown_cluster_0   -> unknown_cluster_0 (unchanged)
    """
    m = re.match(r'^([A-Za-z]+\d+)_.*?(\d+)$', pid)
    if m:
        return f'{m.group(1)}_{m.group(2)}'
    return pid

submission = query_df[['image_id']].copy()
submission['cluster'] = ['cluster_' + clean_cluster_id(pid) for pid in predicted_ids]

dbeps = CFG['dbscan_eps']
out_path = os.path.join(ROOT, f'submission_dinov2_7_dbep{dbeps}.csv')
submission.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

Saved: ../submission_dinov2_7_dbep0.45.csv
